In [1]:
import copy
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

import torch
import torch.nn as nn
from torch.distributions import Categorical

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_AGENTS = 4
HORIZON = 600          # секунд в одном эпизоде
ROLLOUT = 128         # шагов перед обновлением сетей
TRAIN_STEPS = 60_000  # общий бюджет обучения

print("Устройство:", DEVICE)

Устройство: cuda


In [17]:
class TrafficEnv:
    # Расположение перекрёстков:
    # 0 — верхний левый, 1 — верхний правый,
    # 2 — нижний левый, 3 — нижний правый.

    # Направления движения:
    # 0 — север, 1 — восток, 2 — юг, 3 — запад.
    # -1 означает выезд из сети.
    NEXT = np.array([
        [-1,  1,  2, -1],
        [-1, -1,  3,  0],
        [ 0,  3, -1, -1],
        [ 1, -1, -1,  2],
    ])

    CAPACITY = 20
    TRAVEL_TIME = 8
    MIN_GREEN = 10
    CLEARANCE = 3

    def reset(self, seed=0, scenario="balanced"):
        allowed = {
            "low", "balanced", "heavy",
            "horizontal", "burst", "changing",
        }

        if scenario not in allowed:
            raise ValueError(f"Неизвестный сценарий: {scenario}")

        self.rng = np.random.default_rng(seed)
        self.scenario = scenario
        self.t = 0

        # Фаза 0 — север–юг, фаза 1 — восток–запад.
        self.phase = np.zeros(4, dtype=int)
        self.green_age = np.zeros(4, dtype=int)
        self.transition = np.zeros(4, dtype=int)

        # Каждая машина хранится как время её появления в системе.
        self.queues = [
            [deque() for _ in range(4)]
            for _ in range(4)
        ]

        # Машины перед заполненными внешними въездами.
        self.outside = [
            [deque() for _ in range(4)]
            for _ in range(4)
        ]

        # Машины между перекрёстками:
        # (время прибытия, перекрёсток назначения,
        #  направление движения, время появления).
        self.transit = []

        self.boundary = np.array([
            [
                self.NEXT[i, (direction + 2) % 4] == -1
                for direction in range(4)
            ]
            for i in range(4)
        ])

        self.demand_scale = self.rng.uniform(
            0.85, 1.15, size=(4, 4)
        )

        self.generated = 0
        self.completed = 0
        self.wait_sum = 0
        self.travel_sum = 0
        self.queue_sum = 0
        self.max_queue = 0

        return self.observe()

    def queue_lengths(self):
        return np.array([
            [len(queue) for queue in intersection]
            for intersection in self.queues
        ], dtype=int)

    def outside_lengths(self):
        return np.array([
            [len(queue) for queue in intersection]
            for intersection in self.outside
        ], dtype=int)

    def occupancy(self):
        # Вместимость дороги занимают и стоящие,
        # и движущиеся к следующему перекрёстку машины.
        occupied = self.queue_lengths()

        for _, destination, direction, _ in self.transit:
            occupied[destination, direction] += 1

        return occupied

    def rates(self):
        # Среднее число новых машин за секунду на внешний въезд.
        rates = np.full((4, 4), 0.16)

        if self.scenario == "low":
            rates[:] = 0.07

        elif self.scenario == "heavy":
            rates[:] = 0.28

        elif self.scenario == "horizontal":
            rates[:, [0, 2]] = 0.07
            rates[:, [1, 3]] = 0.28

        elif self.scenario == "burst":
            rates[:] = 0.32 if 200 <= self.t < 350 else 0.09

        elif self.scenario == "changing":
            rates[:] = 0.07

            directions = (
                [1, 3] if self.t < HORIZON // 2 else [0, 2]
            )
            rates[:, directions] = 0.28

        return rates * self.demand_scale * self.boundary

    def observe(self):
        queues = self.queue_lengths()
        outside = self.outside_lengths()
        occupied = self.occupancy()

        # Время с появления первой машины каждой очереди в системе.
        oldest = np.zeros((4, 4), dtype=np.float32)

        for i in range(4):
            for direction in range(4):
                queue = self.queues[i][direction]

                if queue:
                    oldest[i, direction] = self.t - queue[0]

        # Порядок признаков совместим с маской в ячейке 3.
        state = np.concatenate([
            queues.flatten() / self.CAPACITY,       # 0–15
            outside.flatten() / self.CAPACITY,      # 16–31
            occupied.flatten() / self.CAPACITY,     # 32–47
            oldest.flatten() / HORIZON,             # 48–63
            self.phase,                            # 64–67
            np.minimum(self.green_age, 60) / 60,    # 68–71
            self.transition / self.CLEARANCE,       # 72–75
            [self.t / HORIZON],                     # 76
        ]).astype(np.float32)

        # Каждый агент видит состояние всей сети.
        return np.tile(state, (4, 1))

    def step(self, actions):
        actions = np.asarray(actions)

        if actions.shape != (4,):
            raise ValueError(
                "Нужно передать по одному действию для 4 агентов."
            )

        if not np.isin(actions, [0, 1]).all():
            raise ValueError("Действия должны быть 0 или 1.")

        if self.t >= HORIZON:
            raise RuntimeError("Эпизод завершён. Выполни reset().")

        switched = np.zeros(4, dtype=np.float32)

        # 0 — сохранить фазу, 1 — запросить переключение.
        for i, action in enumerate(actions):
            can_switch = (
                self.transition[i] == 0
                and self.green_age[i] >= self.MIN_GREEN
            )

            if action == 1 and can_switch:
                self.transition[i] = self.CLEARANCE
                switched[i] = 1

        # Прибытие машин к соседним перекрёсткам.
        still_travelling = []

        for arrival, destination, direction, born in self.transit:
            if arrival <= self.t:
                self.queues[destination][direction].append(born)
            else:
                still_travelling.append(
                    (arrival, destination, direction, born)
                )

        self.transit = still_travelling

        # Генерация случайного потока на внешних въездах.
        arrivals = self.rng.poisson(self.rates())
        self.generated += int(arrivals.sum())

        for i in range(4):
            for direction in range(4):
                count = int(arrivals[i, direction])

                self.outside[i][direction].extend(
                    [self.t] * count
                )

        # Запуск машин с внешних въездов при наличии места.
        occupied = self.occupancy()

        for i in range(4):
            for direction in range(4):
                external_queue = self.outside[i][direction]

                while (
                    external_queue
                    and occupied[i, direction] < self.CAPACITY
                ):
                    born = external_queue.popleft()
                    self.queues[i][direction].append(born)
                    occupied[i, direction] += 1

        # Чередуем порядок обработки перекрёстков.
        processing_order = np.roll(
            np.arange(4), self.t % 4
        )

        for i in processing_order:
            # Во время перехода движение через перекрёсток закрыто.
            if self.transition[i] > 0:
                continue

            directions = (
                [0, 2] if self.phase[i] == 0 else [1, 3]
            )

            for direction in directions:
                queue = self.queues[i][direction]

                if not queue:
                    continue

                destination = self.NEXT[i, direction]

                # Проверяем место на следующей дороге.
                if (
                    destination >= 0
                    and occupied[destination, direction] >= self.CAPACITY
                ):
                    continue

                # За секунду проезжает одна машина
                # из каждого разрешённого направления.
                born = queue.popleft()
                occupied[i, direction] -= 1

                if destination == -1:
                    self.completed += 1
                    self.travel_sum += self.t + 1 - born

                else:
                    # Место на следующей дороге резервируется сразу.
                    occupied[destination, direction] += 1

                    self.transit.append((
                        self.t + self.TRAVEL_TIME,
                        destination,
                        direction,
                        born,
                    ))

        # Очереди после обслуживания за текущую секунду.
        # Учитываем также машины, ожидающие снаружи сети.
        waiting = (
            self.queue_lengths()
            + self.outside_lengths()
        )
        local_wait = waiting.sum(axis=1)

        self.wait_sum += int(waiting.sum())
        self.queue_sum += int(waiting.sum())
        self.max_queue = max(
            self.max_queue,
            int(waiting.max()),
        )

        # Новая функция награды.
        empty_bonus = 0.05 * (local_wait == 0)
        local_penalty = 0.05 * local_wait
        network_penalty = 0.01 * local_wait.mean()
        switch_penalty = 0.05 * switched

        reward = (
            empty_bonus
            - local_penalty
            - network_penalty
            - switch_penalty
        ).astype(np.float32)

        # Обновляем таймеры светофоров.
        for i in range(4):
            if self.transition[i] > 0:
                self.transition[i] -= 1

                if self.transition[i] == 0:
                    self.phase[i] = 1 - self.phase[i]
                    self.green_age[i] = 0

            else:
                self.green_age[i] += 1

        self.t += 1
        done = self.t >= HORIZON

        return self.observe(), reward, done

    def metrics(self):
        return {
            "создано": self.generated,
            "выехало": self.completed,
            "осталось": self.generated - self.completed,
            "ожидание_на_созданную_машину_с":
                self.wait_sum / max(self.generated, 1),
            "средняя_очередь_в_сети":
                self.queue_sum / max(self.t, 1),
            "макс_очередь_одного_въезда":
                self.max_queue,
            "время_поездки_выехавших_с":
                self.travel_sum / max(self.completed, 1),
        }


env = TrafficEnv()
obs = env.reset(seed=SEED)
OBS_SIZE = obs.shape[1]

print("Перекрёстков:", len(obs))
print("Размер наблюдения:", OBS_SIZE)
print("Форма наблюдений:", obs.shape)
print("Новая функция награды подключена.")

Перекрёстков: 4
Размер наблюдения: 77
Форма наблюдений: (4, 77)
Новая функция награды подключена.


In [18]:
from pathlib import Path

# Сохраняем текущую версию перед новым экспериментом.
backup_path = Path("traffic_agents_before_mask.pt")

if "agents" in globals() and not backup_path.exists():
    torch.save({
        "agents": [agent.state_dict() for agent in agents],
        "optimizers": [
            optimizer.state_dict() for optimizer in optimizers
        ],
        "obs_size": OBS_SIZE,
    }, backup_path)
    print("Предыдущие агенты сохранены:", backup_path)


# В текущем observe() первые 64 значения описывают машины.
# Затем идут 4 фазы, 4 возраста зелёного и 4 таймера перехода.
GREEN_AGE_START = 68
TRANSITION_START = 72

if OBS_SIZE != 77:
    raise ValueError(
        "Этот код рассчитан на текущую версию TrafficEnv "
        "с размером наблюдения 77."
    )


class ActorCritic(nn.Module):
    def __init__(self, obs_size, agent_id):
        super().__init__()
        self.agent_id = agent_id

        self.body = nn.Sequential(
            nn.Linear(obs_size, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
        )

        self.actor = nn.Linear(128, 2)
        self.critic = nn.Linear(128, 1)

        # Начинаем с близких вероятностей допустимых действий.
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        nn.init.zeros_(self.actor.bias)

    def forward(self, observations):
        features = self.body(observations)
        logits = self.actor(features)
        value = self.critic(features).squeeze(-1)

        # В observe() возраст зелёного делится на 60.
        green_age = (
            observations[..., GREEN_AGE_START + self.agent_id] * 60
        )
        transition = observations[
            ..., TRANSITION_START + self.agent_id
        ]

        can_switch = (
            (transition == 0)
            & (green_age >= TrafficEnv.MIN_GREEN - 1e-5)
        )

        # Действие 0 допустимо всегда.
        # Действие 1 допустимо только при разрешённом переключении.
        action_mask = torch.stack([
            torch.ones_like(can_switch, dtype=torch.bool),
            can_switch,
        ], dim=-1)

        logits = logits.masked_fill(
            ~action_mask,
            torch.finfo(logits.dtype).min,
        )

        return logits, value


torch.manual_seed(SEED)

agents = [
    ActorCritic(OBS_SIZE, agent_id=i).to(DEVICE)
    for i in range(4)
]

optimizers = [
    torch.optim.Adam(agent.parameters(), lr=3e-4)
    for agent in agents
]


@torch.no_grad()
def predict(observations, greedy=False):
    actions = []
    log_probs = []
    values = []

    for i, agent in enumerate(agents):
        x = torch.as_tensor(
            observations[i],
            dtype=torch.float32,
            device=DEVICE,
        )

        logits, value = agent(x)
        distribution = Categorical(logits=logits)

        if greedy:
            action = logits.argmax(dim=-1)
        else:
            action = distribution.sample()

        actions.append(action.item())
        log_probs.append(distribution.log_prob(action).item())
        values.append(value.item())

    return (
        np.asarray(actions, dtype=int),
        np.asarray(log_probs, dtype=np.float32),
        np.asarray(values, dtype=np.float32),
    )


# Проверяем маску на начальной позиции.
check_env = TrafficEnv()
check_obs = check_env.reset(seed=SEED)
check_actions, _, _ = predict(check_obs)

assert np.all(check_actions == 0), (
    "В начале эпизода переключение должно быть недоступно."
)

print("Созданы четыре новых агента.")
print("Проверка начальной маски пройдена.")

Созданы четыре новых агента.
Проверка начальной маски пройдена.


In [23]:
from pathlib import Path
import time

# Общая цель, включая уже выполненные шаги.
TRAIN_STEPS = 180_000
ROLLOUT = 128
REPORT_EVERY = 1280

GAMMA = 0.99
GAE_LAMBDA = 0.95
PPO_CLIP = 0.2
PPO_EPOCHS = 4
ENTROPY_COEF = 0.01

save_path = Path("traffic_agents_new_reward_continued.pt")
initial_path = Path("traffic_new_reward_60000.pt")

# Предпочитаем последнее сохранение продолженного обучения.
load_path = save_path if save_path.exists() else initial_path

if not load_path.exists():
    raise FileNotFoundError(
        f"Нет сохранения {save_path.name} или {initial_path.name}. "
        "Сначала выполни ячейку сохранения модели на 60 000 шагах."
    )

checkpoint = torch.load(
    load_path,
    map_location=DEVICE,
    weights_only=True,
)

if checkpoint["obs_size"] != OBS_SIZE:
    raise ValueError("Размер наблюдения отличается от сохранённой модели.")

for i in range(4):
    agents[i].load_state_dict(checkpoint["agents"][i])
    optimizers[i].load_state_dict(checkpoint["optimizers"][i])
    agents[i].train()

steps_done = int(checkpoint["steps"])

# При загрузке начинаем новый эпизод.
# Уже обученные веса и состояния оптимизаторов восстановлены.
episode_seed = int(checkpoint.get("next_episode_seed", 1000))
scenario_rng = np.random.default_rng(SEED + steps_done)

training_scenarios = [
    "low",
    "balanced",
    "heavy",
    "horizontal",
]

env = TrafficEnv()
obs = env.reset(
    seed=episode_seed,
    scenario=scenario_rng.choice(training_scenarios),
)

episode_rewards = []
current_reward = 0.0
last_report_step = steps_done
start_time = time.perf_counter()
start_steps = steps_done

print(
    f"Загружен файл {load_path.name}\n"
    f"Выполнено шагов: {steps_done}\n"
    f"Цель: {TRAIN_STEPS}\n"
    f"Осталось: {max(TRAIN_STEPS - steps_done, 0)}",
    flush=True,
)

while steps_done < TRAIN_STEPS:
    states = []
    actions = []
    old_logs = []
    rewards = []
    values = []
    dones = []
    next_values = []

    rollout_length = min(
        ROLLOUT,
        TRAIN_STEPS - steps_done,
    )

    # Собираем опыт всех четырёх агентов.
    for _ in range(rollout_length):
        action, log_prob, value = predict(obs, greedy=False)
        next_obs, reward, done = env.step(action)

        # Оценка следующего состояния для расчёта будущей награды.
        with torch.no_grad():
            bootstrap = []

            for i, agent in enumerate(agents):
                next_tensor = torch.as_tensor(
                    next_obs[i],
                    dtype=torch.float32,
                    device=DEVICE,
                )
                _, next_value = agent(next_tensor)
                bootstrap.append(next_value.item())

        states.append(obs.copy())
        actions.append(action)
        old_logs.append(log_prob)
        rewards.append(reward)
        values.append(value)
        dones.append(done)
        next_values.append(bootstrap)

        current_reward += float(reward.mean())
        steps_done += 1
        obs = next_obs

        if done:
            episode_rewards.append(current_reward)
            current_reward = 0.0

            episode_seed += 1
            obs = env.reset(
                seed=episode_seed,
                scenario=scenario_rng.choice(training_scenarios),
            )

    rewards = np.asarray(rewards, dtype=np.float32)
    values = np.asarray(values, dtype=np.float32)
    next_values = np.asarray(next_values, dtype=np.float32)
    dones = np.asarray(dones, dtype=np.float32)

    # GAE — оценка преимущества выбранных действий.
    advantages = np.zeros_like(rewards)
    last_advantage = np.zeros(4, dtype=np.float32)

    for t in reversed(range(rollout_length)):
        alive = 1.0 - dones[t]

        delta = (
            rewards[t]
            + GAMMA * next_values[t] * alive
            - values[t]
        )

        last_advantage = (
            delta
            + GAMMA * GAE_LAMBDA * alive * last_advantage
        )

        advantages[t] = last_advantage

    returns = advantages + values

    states_t = torch.as_tensor(
        np.asarray(states),
        dtype=torch.float32,
        device=DEVICE,
    )
    actions_t = torch.as_tensor(
        np.asarray(actions),
        dtype=torch.long,
        device=DEVICE,
    )
    old_logs_t = torch.as_tensor(
        np.asarray(old_logs),
        dtype=torch.float32,
        device=DEVICE,
    )
    returns_t = torch.as_tensor(
        returns,
        dtype=torch.float32,
        device=DEVICE,
    )
    advantages_t = torch.as_tensor(
        advantages,
        dtype=torch.float32,
        device=DEVICE,
    )

    # Обновляем каждую сеть по её действиям и наградам.
    for i, (agent, optimizer) in enumerate(zip(agents, optimizers)):
        advantage = advantages_t[:, i]
        advantage = (
            (advantage - advantage.mean())
            / (advantage.std(unbiased=False) + 1e-8)
        )

        for _ in range(PPO_EPOCHS):
            logits, predicted_values = agent(states_t[:, i])
            distribution = Categorical(logits=logits)

            new_logs = distribution.log_prob(actions_t[:, i])
            ratio = (new_logs - old_logs_t[:, i]).exp()

            unclipped = ratio * advantage
            clipped = (
                ratio.clamp(1 - PPO_CLIP, 1 + PPO_CLIP)
                * advantage
            )

            actor_loss = -torch.minimum(
                unclipped, clipped
            ).mean()

            critic_loss = (
                predicted_values - returns_t[:, i]
            ).square().mean()

            loss = (
                actor_loss
                + 0.5 * critic_loss
                - ENTROPY_COEF * distribution.entropy().mean()
            )

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
            optimizer.step()

    # Интервал считаем относительно последнего вывода.
    # Это работает и при старте с 60 000 шагов.
    should_report = (
        steps_done - last_report_step >= REPORT_EVERY
        or steps_done == TRAIN_STEPS
    )

    if should_report:
        elapsed = time.perf_counter() - start_time
        speed = (steps_done - start_steps) / max(elapsed, 1e-8)

        recent_reward = (
            f"{np.mean(episode_rewards[-10:]):.2f}"
            if episode_rewards
            else "пока нет завершённых эпизодов"
        )

        # Сохраняем после завершения обновлений всех четырёх сетей.
        # Сначала пишем временный файл, затем заменяем основной.
        temporary_path = save_path.with_suffix(".tmp")

        torch.save({
            "agents": [
                agent.state_dict()
                for agent in agents
            ],
            "optimizers": [
                optimizer.state_dict()
                for optimizer in optimizers
            ],
            "steps": steps_done,
            "obs_size": OBS_SIZE,
            "next_episode_seed": episode_seed + 1,
        }, temporary_path)

        temporary_path.replace(save_path)

        print(
            f"Шаг {steps_done}/{TRAIN_STEPS} | "
            f"Эпизодов в этом запуске: {len(episode_rewards)} | "
            f"Средняя награда: {recent_reward} | "
            f"Скорость: {speed:.1f} шаг/с | Сохранено",
            flush=True,
        )

        last_report_step = steps_done

print(
    f"\nОбучение завершено. Выполнено шагов: {steps_done}.",
    flush=True,
)

Загружен файл traffic_new_reward_60000.pt
Выполнено шагов: 60000
Цель: 180000
Осталось: 120000
Шаг 61280/180000 | Эпизодов в этом запуске: 2 | Средняя награда: -226.41 | Скорость: 90.7 шаг/с | Сохранено
Шаг 62560/180000 | Эпизодов в этом запуске: 4 | Средняя награда: -207.58 | Скорость: 106.2 шаг/с | Сохранено
Шаг 63840/180000 | Эпизодов в этом запуске: 6 | Средняя награда: -192.09 | Скорость: 111.1 шаг/с | Сохранено
Шаг 65120/180000 | Эпизодов в этом запуске: 8 | Средняя награда: -188.15 | Скорость: 113.3 шаг/с | Сохранено
Шаг 66400/180000 | Эпизодов в этом запуске: 10 | Средняя награда: -186.43 | Скорость: 114.8 шаг/с | Сохранено
Шаг 67680/180000 | Эпизодов в этом запуске: 12 | Средняя награда: -184.18 | Скорость: 115.7 шаг/с | Сохранено
Шаг 68960/180000 | Эпизодов в этом запуске: 14 | Средняя награда: -195.19 | Скорость: 117.1 шаг/с | Сохранено
Шаг 70240/180000 | Эпизодов в этом запуске: 17 | Средняя награда: -220.53 | Скорость: 118.0 шаг/с | Сохранено
Шаг 71520/180000 | Эпизодов в 

In [24]:
def run_evaluation(scenario, seed, controller, record=False):
    test_env = TrafficEnv()
    observation = test_env.reset(seed=seed, scenario=scenario)
    frames = []

    for _ in range(HORIZON):
        if controller == "PPO":
            action, _, _ = predict(observation, greedy=False)
        else:
            action = (
                (test_env.green_age >= 20)
                & (test_env.transition == 0)
            ).astype(int)

        observation, _, done = test_env.step(action)

        if record and test_env.t % 3 == 0:
            frames.append({
                "time": test_env.t,
                "queues": test_env.queue_lengths().copy(),
                "outside": test_env.outside_lengths().copy(),
                "phase": test_env.phase.copy(),
                "transition": test_env.transition.copy(),
                "transit": copy.deepcopy(test_env.transit),
                "completed": test_env.completed,
            })

        if done:
            break

    return test_env.metrics(), frames


scenarios = ["low", "balanced", "heavy", "horizontal", "burst", "changing"]
rows = []

for scenario in scenarios:
    for seed in range(10_000, 10_005):
        for controller in ["Фиксированный цикл", "PPO"]:
            metrics, _ = run_evaluation(scenario, seed, controller)
            rows.append({
                "сценарий": scenario,
                "управление": controller,
                "seed": seed,
                **metrics,
            })

results = pd.DataFrame(rows)

summary = results.groupby(["сценарий", "управление"])[[
    "выехало",
    "осталось",
    "ожидание_на_созданную_машину_с",
    "средняя_очередь_в_сети",
]].mean().round(2)

display(summary)
results.to_csv("traffic_agents_new_reward_continued.csv", index=False)

выехало  осталось  \
сценарий   управление                              
balanced   PPO                   767.2      24.2   
           Фиксированный цикл    764.8      26.6   
burst      PPO                   702.6      10.6   
           Фиксированный цикл    700.6      12.6   
changing   PPO                   824.6      35.6   
           Фиксированный цикл    817.8      42.4   
heavy      PPO                  1306.4      51.2   
           Фиксированный цикл   1307.0      50.6   
horizontal PPO                   806.4      38.4   
           Фиксированный цикл    814.4      30.4   
low        PPO                   330.0      11.4   
           Фиксированный цикл    328.6      12.8   

                               ожидание_на_созданную_машину_с  \
сценарий   управление                                           
balanced   PPO                                          13.62   
           Фиксированный цикл                           13.22   
burst      PPO                                          16.42   
           Фиксированный цикл                           16.13   
changing   PPO                                          17.63   
           Фиксированный цикл                           15.86   
heavy      PPO                                          17.59   
           Фиксированный цикл                           16.58   
horizontal PPO                                          17.21   
           Фиксированный цикл                           15.87   
low        PPO                                          11.90   
           Фиксированный цикл                           11.46   

                               средняя_очередь_в_сети  
сценарий   управление                                  
balanced   PPO                                  17.96  
           Фиксированный цикл                   17.44  
burst      PPO                                  19.50  
           Фиксированный цикл                   19.18  
changing   PPO                                  25.35  
           Фиксированный цикл                   22.78  
heavy      PPO                                  39.83  
           Фиксированный цикл                   37.54  
horizontal PPO                                  24.23  
           Фиксированный цикл                   22.37  
low        PPO                                   6.77  
           Фиксированный цикл                    6.51

In [25]:
_, frames = run_evaluation(
    scenario="changing",
    seed=20_000,
    controller="PPO",
    record=True,
)

centers = np.array([
    [0.0, 1.0], [1.0, 1.0],
    [0.0, 0.0], [1.0, 0.0],
])

# Вектор направления движения: север, восток, юг, запад.
vectors = np.array([[0, 1], [1, 0], [0, -1], [-1, 0]])
links = [(0, 1), (0, 2), (1, 3), (2, 3)]

fig, ax = plt.subplots(figsize=(9, 9))

def draw_frame(index):
    ax.clear()
    frame = frames[index]

    for coordinate in [0, 1]:
        ax.plot(
            [-0.5, 1.5], [coordinate, coordinate],
            color="#dadada", linewidth=32, zorder=0
        )
        ax.plot(
            [coordinate, coordinate], [-0.5, 1.5],
            color="#dadada", linewidth=32, zorder=0
        )

    for i, (x, y) in enumerate(centers):
        phase = frame["phase"][i]
        transition = frame["transition"][i]

        ns_color = "green" if phase == 0 and transition == 0 else "red"
        ew_color = "green" if phase == 1 and transition == 0 else "red"

        ax.plot([x, x], [y - 0.07, y + 0.07],
                color=ns_color, linewidth=7)
        ax.plot([x - 0.07, x + 0.07], [y, y],
                color=ew_color, linewidth=7)

        for d, direction in enumerate(vectors):
            count = frame["queues"][i, d]
            lateral = np.array([direction[1], -direction[0]]) * 0.035

            for car in range(count):
                distance = 0.11 + car * 0.014
                position = centers[i] - direction * distance + lateral
                ax.scatter(
                    position[0], position[1],
                    s=20, marker="s", color="#2563eb", zorder=3
                )

        queue = frame["queues"][i].sum()
        outside = frame["outside"][i].sum()
        ax.text(
            x + 0.12, y + 0.15,
            f"Узел {i + 1}\nОчередь {queue}\nЗа въездом {outside}",
            fontsize=9,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none")
        )

    for first, second in links:
        travelling = 0
        for _, destination, direction, _ in frame["transit"]:
            source = TrafficEnv.NEXT[destination, (direction + 2) % 4]
            if {int(source), int(destination)} == {first, second}:
                travelling += 1

        midpoint = (centers[first] + centers[second]) / 2
        ax.text(
            midpoint[0], midpoint[1],
            f"В пути {travelling}",
            ha="center", va="center", fontsize=9,
            bbox=dict(facecolor="white", alpha=0.9, edgecolor="none")
        )

    ax.set_title(
        f"PPO · {frame['time']} с · Выехало {frame['completed']} машин\n"
        "Смена основного потока на 300-й секунде"
    )
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_aspect("equal")
    ax.axis("off")


animation = FuncAnimation(
    fig, draw_frame, frames=len(frames), interval=100
)
plt.close(fig)
display(HTML(animation.to_jshtml()))

In [12]:
def diagnose_controller(scenario, seed, controller):
    test_env = TrafficEnv()
    observation = test_env.reset(seed=seed, scenario=scenario)

    switches = np.zeros(4, dtype=int)
    ignored_requests = np.zeros(4, dtype=int)
    waiting_sum = np.zeros(4, dtype=float)
    green_seconds = np.zeros((4, 2), dtype=int)
    transition_seconds = np.zeros(4, dtype=int)

    # Длительности полностью завершённых зелёных фаз.
    completed_green = [[] for _ in range(4)]

    for _ in range(HORIZON):
        can_switch = (
            (test_env.transition == 0)
            & (test_env.green_age >= test_env.MIN_GREEN)
        )

        if controller == "PPO":
            actions, _, _ = predict(observation, greedy=True)
        else:
            actions = (
                (test_env.green_age >= 20)
                & (test_env.transition == 0)
            ).astype(int)

        requested = actions == 1
        accepted = requested & can_switch

        ignored_requests += (requested & ~can_switch).astype(int)
        switches += accepted.astype(int)

        # Запрос переключения запускает переход уже в текущую секунду.
        in_transition = (test_env.transition > 0) | accepted
        transition_seconds += in_transition.astype(int)

        for i in range(4):
            if accepted[i]:
                completed_green[i].append(int(test_env.green_age[i]))

            if not in_transition[i]:
                green_seconds[i, test_env.phase[i]] += 1

        observation, _, done = test_env.step(actions)

        waiting = (
            test_env.queue_lengths()
            + test_env.outside_lengths()
        )
        waiting_sum += waiting.sum(axis=1)

        if done:
            break

    rows = []

    for i in range(4):
        durations = completed_green[i]

        rows.append({
            "сценарий": scenario,
            "управление": controller,
            "seed": seed,
            "перекрёсток": i + 1,
            "переключений": switches[i],
            "недоступных_запросов": ignored_requests[i],
            "средний_зелёный_с":
                float(np.mean(durations)) if durations else np.nan,
            "макс_завершённый_зелёный_с":
                max(durations) if durations else np.nan,
            "зелёный_СЮ_с": green_seconds[i, 0],
            "зелёный_ВЗ_с": green_seconds[i, 1],
            "переходы_с": transition_seconds[i],
            "средняя_очередь": waiting_sum[i] / test_env.t,
            "возраст_последнего_зелёного_с": test_env.green_age[i],
        })

    return rows


diagnostic_rows = []

for scenario in ["balanced", "heavy", "horizontal"]:
    for seed in range(10_000, 10_005):
        for controller in ["Фиксированный цикл", "PPO"]:
            diagnostic_rows.extend(
                diagnose_controller(scenario, seed, controller)
            )

diagnostics = pd.DataFrame(diagnostic_rows)

columns = [
    "переключений",
    "недоступных_запросов",
    "средний_зелёный_с",
    "зелёный_СЮ_с",
    "зелёный_ВЗ_с",
    "переходы_с",
    "средняя_очередь",
]

diagnostic_summary = (
    diagnostics
    .groupby(["сценарий", "управление", "перекрёсток"])[columns]
    .mean()
    .round(2)
)

display(diagnostic_summary)

diagnostics.to_csv("traffic_diagnostics2.csv", index=False)
print("Сохранён файл traffic_diagnostics.csv")

переключений  недоступных_запросов  \
сценарий   управление         перекрёсток                                       
balanced   PPO                1                    45.0                   0.0   
                              2                    45.8                   0.0   
                              3                    45.8                   0.0   
                              4                    44.0                   0.0   
           Фиксированный цикл 1                    26.0                   0.0   
                              2                    26.0                   0.0   
                              3                    26.0                   0.0   
                              4                    26.0                   0.0   
heavy      PPO                1                    46.0                   0.0   
                              2                    46.0                   0.0   
                              3                    46.0                   0.0   
                              4                    46.0                   0.0   
           Фиксированный цикл 1                    26.0                   0.0   
                              2                    26.0                   0.0   
                              3                    26.0                   0.0   
                              4                    26.0                   0.0   
horizontal PPO                1                    45.6                   0.0   
                              2                    45.2                   0.0   
                              3                    45.6                   0.0   
                              4                    45.6                   0.0   
           Фиксированный цикл 1                    26.0                   0.0   
                              2                    26.0                   0.0   
                              3                    26.0                   0.0   
                              4                    26.0                   0.0   

                                           средний_зелёный_с  зелёный_СЮ_с  \
сценарий   управление         перекрёсток                                    
balanced   PPO                1                        10.17         233.8   
                              2                        10.07         232.4   
                              3                        10.00         231.6   
                              4                        10.32         237.6   
           Фиксированный цикл 1                        20.00         262.0   
                              2                        20.00         262.0   
                              3                        20.00         262.0   
                              4                        20.00         262.0   
heavy      PPO                1                        10.00         232.0   
                              2                        10.00         232.0   
                              3                        10.00         232.0   
                              4                        10.00         232.0   
           Фиксированный цикл 1                        20.00         262.0   
                              2                        20.00         262.0   
                              3                        20.00         262.0   
                              4                        20.00         262.0   
horizontal PPO                1                        10.04         231.2   
                              2                        10.12         230.4   
                              3                        10.04         230.8   
                              4                        10.07         230.4   
           Фиксированный цикл 1                        20.00         262.0   
                              2                        20.00         262.0   
                              3                        20.00    

Сохранён файл traffic_diagnostics.csv
